## 🧑‍💻 THÀNH VIÊN 1: DATA CLEANER - Phan Hoàng Quốc Huy (23120048)
Nhiệm vụ: Làm sạch dữ liệu cơ bản (Cấu trúc dữ liệu)
- Đọc và gom nhóm tập dữ liệu.
- Khám phá cấu trúc sơ bộ.
- Phát hiện và xử lý các giá trị rỗng (Missing values / Null).
- Phát hiện và xử lý dữ liệu trùng lặp (Duplicates).
- Bàn giao dataframe đã sạch lỗi cấu trúc cho Thành viên 2.

In [1]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

## PHẦN 1: ĐỌC VÀ KHÁM PHÁ SƠ BỘ DỮ LIỆU
Đọc 3 tập dữ liệu `train.csv`, `dev.csv`, `test.csv` và in ra các thống kê cơ bản để nắm bắt tình trạng của dữ liệu trước khi làm sạch.

In [2]:
def load_datasets(data_dir='.'):
    """
    Đọc 3 file CSV: train.csv, dev.csv, test.csv
    Returns: dict chứa 3 DataFrame
    """
    datasets = {}
    for name in ['train', 'dev', 'test']:
        filepath = os.path.join(data_dir, f'{name}.csv')
        if os.path.exists(filepath):
            df = pd.read_csv(filepath)
            datasets[name] = df
            print(f"📂 {name}.csv: {df.shape[0]} dòng, {df.shape[1]} cột")
            print(f"   Columns: {df.columns.tolist()}")
            print(f"   Dtypes: {df.dtypes.to_dict()}")
            print(f"   Null values: {df.isnull().sum().to_dict()}")
            print(f"   Duplicates: {df.duplicated().sum()}")
            if 'label_id' in df.columns:
                print(f"   Label distribution: {df['label_id'].value_counts().sort_index().to_dict()}")
            print("-" * 50)
        else:
            print(f"❌ Không tìm thấy file {filepath}")
    return datasets

# Đọc dữ liệu
DATA_DIR = '../data' # Thay đổi đường dẫn nếu cần
datasets = load_datasets(DATA_DIR)

📂 train.csv: 24048 dòng, 2 cột
   Columns: ['free_text', 'label_id']
   Dtypes: {'free_text': dtype('O'), 'label_id': dtype('int64')}
   Null values: {'free_text': 2, 'label_id': 0}
   Duplicates: 1356
   Label distribution: {0: 19886, 1: 1606, 2: 2556}
--------------------------------------------------
📂 dev.csv: 2672 dòng, 2 cột
   Columns: ['free_text', 'label_id']
   Dtypes: {'free_text': dtype('O'), 'label_id': dtype('int64')}
   Null values: {'free_text': 0, 'label_id': 0}
   Duplicates: 21
   Label distribution: {0: 2190, 1: 212, 2: 270}
--------------------------------------------------
📂 test.csv: 6680 dòng, 2 cột
   Columns: ['free_text', 'label_id']
   Dtypes: {'free_text': dtype('O'), 'label_id': dtype('int64')}
   Null values: {'free_text': 0, 'label_id': 0}
   Duplicates: 98
   Label distribution: {0: 5548, 1: 444, 2: 688}
--------------------------------------------------


## PHẦN 2: XỬ LÝ DỮ LIỆU THIẾU (NULL) VÀ TRÙNG LẶP (DUPLICATES)
Đảm bảo dataframe không có dòng nào bị trống nội dung (null) và loại bỏ các dòng bị trùng lặp hoàn toàn để tránh nhiễu mô hình.

In [3]:
def handle_missing_values(df, name='dataset'):
    """
    Xử lý dữ liệu thiếu:
    - Với free_text null: xóa dòng (vì không thể phân loại văn bản rỗng)
    """
    null_count = df['free_text'].isnull().sum()
    print(f"🔍 [{name}] Số dòng có free_text null: {null_count}")
    
    if null_count > 0:
        df = df.dropna(subset=['free_text'])
        print(f"   ✅ Đã xóa {null_count} dòng null. Còn lại: {df.shape[0]} dòng")
    
    return df

def handle_duplicates(df, name='dataset'):
    """
    Xử lý dữ liệu trùng lặp:
    - Xóa các dòng hoàn toàn giống nhau (cả text lẫn label)
    """
    dup_count = df.duplicated().sum()
    print(f"🔍 [{name}] Số dòng trùng lặp: {dup_count}")
    
    if dup_count > 0:
        df = df.drop_duplicates()
        print(f"   ✅ Đã xóa {dup_count} dòng trùng lặp. Còn lại: {df.shape[0]} dòng")
    
    return df

## PHẦN 3: CHẠY PIPELINE LÀM SẠCH VÀ LƯU FILE BÀN GIAO
Áp dụng các hàm trên cho cả 3 tập dữ liệu và lưu ra các file `*_structural_clean.csv` để Thành viên 2 (Data Engineer) có thể bắt đầu xử lý text.

In [4]:
OUTPUT_DIR = '../data' # Thay đổi thư mục lưu nếu cần
datasets_clean = {}

print("=" * 60)
print("🚀 BẮT ĐẦU LÀM SẠCH CẤU TRÚC DỮ LIỆU")
print("=" * 60)

for name, df in datasets.items():
    print(f"\n⏳ Đang xử lý tập: {name.upper()}")
    # 1. Xử lý Missing values
    df_clean = handle_missing_values(df, name)
    
    # 2. Xử lý Duplicates
    df_clean = handle_duplicates(df_clean, name)
    
    # Lưu vào dictionary
    datasets_clean[name] = df_clean
    
    # 3. LƯU FILE BÀN GIAO
    output_path = os.path.join(OUTPUT_DIR, f'{name}_structural_clean.csv')
    df_clean.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"   💾 Đã lưu file bàn giao: {output_path}")

print(f"\n{'='*60}")
print("🎯 TỔNG KẾT BƯỚC LÀM SẠCH CƠ BẢN")
print(f"{'='*60}")
for name in datasets_clean.keys():
    orig = datasets[name].shape[0]
    clean = datasets_clean[name].shape[0]
    removed = orig - clean
    print(f"📊 {name}: {orig} dòng → {clean} dòng (Đã lọc {removed} dòng lỗi)")

print("\n🏁 HOÀN TẤT NHIỆM VỤ THÀNH VIÊN 1. SẴN SÀNG BÀN GIAO CHO THÀNH VIÊN 2!")

🚀 BẮT ĐẦU LÀM SẠCH CẤU TRÚC DỮ LIỆU

⏳ Đang xử lý tập: TRAIN
🔍 [train] Số dòng có free_text null: 2
   ✅ Đã xóa 2 dòng null. Còn lại: 24046 dòng
🔍 [train] Số dòng trùng lặp: 1356
   ✅ Đã xóa 1356 dòng trùng lặp. Còn lại: 22690 dòng
   💾 Đã lưu file bàn giao: ../data/train_structural_clean.csv

⏳ Đang xử lý tập: DEV
🔍 [dev] Số dòng có free_text null: 0
🔍 [dev] Số dòng trùng lặp: 21
   ✅ Đã xóa 21 dòng trùng lặp. Còn lại: 2651 dòng
   💾 Đã lưu file bàn giao: ../data/dev_structural_clean.csv

⏳ Đang xử lý tập: TEST
🔍 [test] Số dòng có free_text null: 0
🔍 [test] Số dòng trùng lặp: 98
   ✅ Đã xóa 98 dòng trùng lặp. Còn lại: 6582 dòng
   💾 Đã lưu file bàn giao: ../data/test_structural_clean.csv

🎯 TỔNG KẾT BƯỚC LÀM SẠCH CƠ BẢN
📊 train: 24048 dòng → 22690 dòng (Đã lọc 1358 dòng lỗi)
📊 dev: 2672 dòng → 2651 dòng (Đã lọc 21 dòng lỗi)
📊 test: 6680 dòng → 6582 dòng (Đã lọc 98 dòng lỗi)

🏁 HOÀN TẤT NHIỆM VỤ THÀNH VIÊN 1. SẴN SÀNG BÀN GIAO CHO THÀNH VIÊN 2!
